# Retrieval-Augmented Generation (RAG) System

## Objective

This notebook develops an end-to-end Retrieval-Augmented Generation (RAG) pipeline using open-source Large Language Models (LLMs) and LangChain.

The system performs the following tasks:

- Loads banking and regulatory PDF documents.
- Splits documents into overlapping text chunks.
- Converts each chunk into vector embeddings.
- Stores embeddings in a FAISS vector database.
- Retrieves the most relevant context using semantic similarity.
- Generates grounded responses using an open-source LLM.
- Implements basic governance controls, including prompt injection detection and simple guardrails.

This notebook focuses on **building the RAG pipeline**. Evaluation metrics and independent model validation are performed separately in dedicated notebooks.


# 1. Install Required Libraries

The following libraries are required to build the Retrieval-Augmented Generation (RAG) pipeline.

These include libraries for:

- LLM orchestration
- Document loading
- Text embeddings
- Vector database creation
- PDF processing
- Efficient model inference

In [ ]:
!pip -q install langchain
!pip -q install langchain-community
!pip -q install langchain-huggingface
!pip -q install transformers
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install pypdf
!pip -q install accelerate
!pip -q install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


# 2. Import Required Libraries

This section imports the libraries required for document processing, text embeddings, vector database creation, and the overall RAG pipeline.

Each library serves a specific purpose:

- **LangChain** – Orchestrates the RAG workflow.
- **PyPDFLoader** – Loads PDF documents.
- **Hugging Face Embeddings** – Converts text into vector embeddings.
- **FAISS** – Stores embeddings for efficient similarity search.
- **Transformers & Sentence Transformers** – Provide access to pre-trained transformer models.

In [ ]:
import langchain
import transformers
import sentence_transformers

from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Everything is working!")

/tmp/ipykernel_1107/46035485.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Everything is working!


# 3. Load Source Documents

The RAG system begins by loading the source documents that will form its knowledge base.

In this project, banking and regulatory PDF documents are loaded using LangChain's `PyPDFLoader`.

Each PDF is converted into LangChain `Document` objects, where every page is stored separately along with useful metadata such as the source document name and page number.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Basel Norms.pdf to Basel Norms (2).pdf
Saving HDFC_Bank_Annual_Report_2024_25-310202.pdf to HDFC_Bank_Annual_Report_2024_25-310202 (2).pdf


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir():
    if file.endswith(".pdf"):
        print(f"Loading: {file}")
        loader = PyPDFLoader(file)
        documents.extend(loader.load())

print(f"\nTotal pages loaded: {len(documents)}")

Loading: HDFC_Bank_Annual_Report_2024_25-310202 (1).pdf
Loading: HDFC_Bank_Annual_Report_2024_25-310202 (2).pdf
Loading: HDFC_Bank_Annual_Report_2024_25-310202.pdf
Loading: Basel Norms (1).pdf
Loading: Basel Norms (2).pdf
Loading: Basel Norms.pdf

Total pages loaded: 2631


In [ ]:
print(documents[10].page_content[:1000])

The Indian economy, despite global 
headwinds, has demonstrated great 
resilience with stable and predictable 
policy actions in the ﬁscal and monetary 
domain that ensured we continue to 
sustain the growth momentum coming 
out of the shocks of the preceding years. 
The economy grew at 6.5 per cent in 
the ﬁscal year 2024-25 with inﬂation 
moderating to 4.6 per cent, performing 
under a shifting domestic policy stance 
that transitioned from a “Withdrawal of 
Accommodation” to “Accommodative” 
and back to “Neutral” recently. Our 
regulator has been proactive in its policy 
response, with inﬂation under reasonable 
control, despite pressures coming in 
from a sharp depreciation of the rupee. 
It has rationalised regulatory burden and 
removed restriction on ﬁnancial sectors 
through its policy actions. The national 
economic policy regime has stepped up 
to stabilise the economy, adhering to its 
stated objective of balancing the budget 
and keeping the ﬁscal deﬁcit within 
limits ther

# 4. Split Documents into Chunks

Large Language Models have a limited context window and cannot efficiently process entire documents at once.

To address this, the source documents are divided into smaller overlapping chunks using LangChain's `RecursiveCharacterTextSplitter`.

**Configuration used:**

- **Chunk Size (800):** Maximum number of characters in each chunk.
- **Chunk Overlap (150):** Preserves context between adjacent chunks by repeating 150 characters.

Overlapping chunks help prevent important information from being split across chunk boundaries, improving retrieval quality.

In [ ]:
!pip -q install -U langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 12246


# 5. Load the Embedding Model

To perform semantic search, the text chunks must first be converted into numerical vector representations called **embeddings**.

This project uses the **BAAI/bge-small-en-v1.5** embedding model from Hugging Face.

The embedding model converts each text chunk into a high-dimensional vector, allowing semantically similar text to be located using vector similarity rather than exact keyword matching.

The embeddings are generated on the GPU (when available) and normalized to improve the consistency of similarity calculations.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


# 6. Build the FAISS Vector Database

Once the document chunks have been converted into embeddings, they are stored in a **FAISS (Facebook AI Similarity Search)** vector database.

Unlike a traditional SQL database that searches using exact matches, a vector database stores numerical embeddings and retrieves information based on **semantic similarity**.

This enables the RAG system to identify the most relevant document chunks even when the user's query is phrased differently from the source text.

In [ ]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


# 7. Configure the Retriever

The retriever is responsible for identifying the most relevant document chunks for a user's query.

When a question is received, the retriever:

1. Converts the query into an embedding.
2. Searches the FAISS vector database using cosine similarity.
3. Retrieves the **Top-K** most semantically similar document chunks.

These retrieved chunks are then supplied to the Large Language Model as supporting context for answer generation.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever created successfully!")

Retriever created successfully!


In [ ]:
query = "What is operational risk?"

docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} documents\n")

print(docs[0].page_content)

Retrieved 4 documents

- 105 - 
 
9.      Capital Charge for Operational Risk 
9.1    Definition of Operational Risk 
Operational risk is defined as the risk of loss resulting from inadequate or failed internal 
processes, people and systems  or from external events. This definition includes legal risk, 
but excludes strategic and reputational risk. Legal risk includes, but is not limited to, 
exposure to fines, penalties, or punitive damages resulting from supervisory actions, as well 
as private settlements. 
 
9.2     The Measurement Methodologies 
9.2.1 The New Capital Adequacy Framework outlines three methods for calculating 
operational risk capital charges in a continuum of increasing sophistication and risk 
sensitivity: (i) the Basic Indicator Approach (BIA); (ii) the Standardised Approach (TSA); and


# 8. Load the Large Language Model (LLM)

The retrieved document chunks alone cannot answer a user's question.

A Large Language Model (LLM) is used to read:

- The user's question
- The retrieved document context

and generate a natural language response grounded in the retrieved information.

This project uses **Microsoft Phi-3 Mini 4K Instruct**, an open-source instruction-tuned language model suitable for local inference and RAG applications.

In [ ]:
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    device=0,
    pipeline_kwargs={
        "max_new_tokens": 200,
        "temperature": 0.0,
        "do_sample": False,
    },
)

print("LLM Loaded!")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM Loaded!


In [ ]:
print(llm.invoke("What is operational risk?"))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


What is operational risk?

Operational risk refers to the risk of loss resulting from inadequate or failed internal processes, people, systems, or from external events. This type of risk is inherent in the day-to-day operations of a business and can arise from a variety of sources, including:

1. **Internal Processes**: This includes risks associated with the failure of internal procedures, systems, or controls. For example, a company might have a process for approving financial transactions, but if this process is not followed correctly, it could lead to financial loss.

2. **People**: Human error is a significant source of operational risk. This can include mistakes made by employees, fraud, or even the loss of key personnel. For instance, an employee might accidentally input incorrect data into a system, leading to inaccurate financial reporting.

3. **Systems**: Technological failures or cybersecurity breaches


#9. Generate and Export Results

In this section, the RAG pipeline is executed using the predefined set of benchmark questions. For each question, the system retrieves the most relevant document chunks and generates a response based on the retrieved context.

The generated responses, along with the retrieved context and corresponding reference answers, are consolidated into a structured dataset. The final results are then exported as CSV files, providing a complete record of the model outputs for subsequent analysis and reporting.

In [ ]:
questions = [
    "What is operational risk?",
    "What is legal risk?",
    "What are the three operational risk measurement approaches?",
    "What is the Basic Indicator Approach?",
    "What is the Standardised Approach?",
    "What is the Advanced Measurement Approach?",
    "How does HDFC define operational risk?",
    "What risks are excluded from operational risk?",
    "What is Basel III?",
    "What is CET1 capital?",
    "What is Tier 1 capital?",
    "What is Tier 2 capital?",
    "How does HDFC manage operational risk?",
    "What is market risk?",
    "What is credit risk?",
    "How are operational losses managed?",
    "What is the role of internal controls?",
    "What is the role of risk governance?",
    "What are external events in operational risk?",
    "What is the objective of Basel norms?"
]

print(f"Questions loaded: {len(questions)}")

Questions loaded: 20


In [ ]:
results = []

for question in questions:

    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a banking risk assistant.

Answer ONLY using the information provided in the context.

If the answer is not present in the context, say:
Answer not found.

Context:
{context}

Question:
{question}

Answer:
"""

    raw_output = llm.invoke(prompt)

    # Remove the prompt if the model echoes it
    if "Answer:" in raw_output:
        answer = raw_output.split("Answer:")[-1].strip()
    else:
        answer = raw_output.strip()

    results.append({
        "question": question,
        "context": context,
        "answer": answer
    })

print(f"Generated {len(results)} answers.")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Generated 20 answers.


In [ ]:
import pandas as pd

df = pd.DataFrame(results)

df.head()

,question,context,answer
0,What is operational risk?,- 105 - \n \n9. Capital Charge for Operat...,Operational risk is defined as the risk of los...
1,What is legal risk?,• Strategic Risk\n• Compliance Risk\n• Reputat...,Legal risk is not explicitly mentioned in the ...
2,What are the three operational risk measuremen...,9.1 Definition of Operational Risk \n 9.2 The...,The Supervisory Review
3,What is the Basic Indicator Approach?,(c) Alpha = 15 per cent \n \n \n9.3.4 As a poi...,Banks using the Basic Indicator Approach are e...
4,What is the Standardised Approach?,risks are based on increasing risk sensitivity...,The Standardised Approach is one of the option...


In [ ]:
references = {
    "What is operational risk?":
        "Operational risk is the risk of loss resulting from inadequate or failed internal processes, people and systems, or from external events. It includes legal risk but excludes strategic and reputational risk.",

    "What is legal risk?":
        "Legal risk includes exposure to fines, penalties, punitive damages, supervisory actions and private settlements.",

    "What are the three operational risk measurement approaches?":
        "The three approaches are the Basic Indicator Approach (BIA), the Standardised Approach (TSA) and the Advanced Measurement Approach (AMA).",

    "What is the Basic Indicator Approach?":
        "The Basic Indicator Approach calculates operational risk capital using a fixed percentage of the bank's gross income.",

    "What is the Standardised Approach?":
        "The Standardised Approach calculates operational risk capital by applying different percentages to different business lines.",

    "What is the Advanced Measurement Approach?":
        "The Advanced Measurement Approach allows banks to use internally developed risk measurement systems subject to regulatory approval.",

    "How does HDFC define operational risk?":
        "HDFC defines operational risk as the risk arising from inadequate or failed internal processes, people, systems or external events, including legal risk but excluding strategic and reputational risk.",

    "What risks are excluded from operational risk?":
        "Strategic risk and reputational risk are excluded from the definition of operational risk.",

    "What is Basel III?":
        "Basel III is an international regulatory framework designed to strengthen bank capital, improve risk management and enhance financial stability.",

    "What is CET1 capital?":
        "Common Equity Tier 1 (CET1) capital primarily consists of common shares, retained earnings and other disclosed reserves after regulatory adjustments."
}

In [ ]:
df["ground_truth"] = df["question"].map(references)

df.head()

,question,context,answer,ground_truth
0,What is operational risk?,- 105 - \n \n9. Capital Charge for Operat...,Operational risk is defined as the risk of los...,Operational risk is the risk of loss resulting...
1,What is legal risk?,• Strategic Risk\n• Compliance Risk\n• Reputat...,Legal risk is not explicitly mentioned in the ...,"Legal risk includes exposure to fines, penalti..."
2,What are the three operational risk measuremen...,9.1 Definition of Operational Risk \n 9.2 The...,The Supervisory Review,The three approaches are the Basic Indicator A...
3,What is the Basic Indicator Approach?,(c) Alpha = 15 per cent \n \n \n9.3.4 As a poi...,Banks using the Basic Indicator Approach are e...,The Basic Indicator Approach calculates operat...
4,What is the Standardised Approach?,risks are based on increasing risk sensitivity...,The Standardised Approach is one of the option...,The Standardised Approach calculates operation...


In [ ]:
import pandas as pd

# If df already exists, this will just save it
df.to_csv("rag_results.csv", index=False)

print(df.shape)
print("Saved rag_results.csv successfully!")

(20, 4)
Saved rag_results.csv successfully!


In [ ]:
from google.colab import files

files.download("rag_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
references.update({

    "What is Tier 1 capital?":
        "Tier 1 capital consists primarily of Common Equity Tier 1 capital and Additional Tier 1 capital. It represents a bank's core capital used to absorb losses while continuing operations.",

    "What is Tier 2 capital?":
        "Tier 2 capital consists of supplementary capital such as subordinated debt and other eligible instruments that absorb losses during winding up.",

    "How does HDFC manage operational risk?":
        "HDFC manages operational risk through governance frameworks, internal controls, risk assessments, monitoring, mitigation measures and regulatory compliance.",

    "What is market risk?":
        "Market risk is the risk of losses arising from movements in market prices such as interest rates, foreign exchange rates, equity prices and commodity prices.",

    "What is credit risk?":
        "Credit risk is the risk of financial loss arising from a borrower or counterparty failing to meet its contractual obligations.",

    "How are operational losses managed?":
        "Operational losses are managed through risk identification, monitoring, internal controls, mitigation strategies, incident reporting and continuous improvement of processes.",

    "What is the role of internal controls?":
        "Internal controls help prevent, detect and mitigate operational risks by ensuring effective processes, compliance and oversight.",

    "What is the role of risk governance?":
        "Risk governance establishes oversight, accountability and policies for identifying, monitoring and managing risks across the organization.",

    "What are external events in operational risk?":
        "External events include incidents outside the organization's control such as natural disasters, cyber attacks, fraud, terrorism or other disruptions that can cause operational losses.",

    "What is the objective of Basel norms?":
        "The objective of Basel norms is to strengthen banks' capital adequacy, improve risk management practices and enhance the stability of the financial system."

})

In [ ]:
df["ground_truth"] = df["question"].map(references)

df.head(20)

,question,context,answer,ground_truth
0,What is operational risk?,- 105 - \n \n9. Capital Charge for Operat...,Operational risk is defined as the risk of los...,Operational risk is the risk of loss resulting...
1,What is legal risk?,• Strategic Risk\n• Compliance Risk\n• Reputat...,Legal risk is not explicitly mentioned in the ...,"Legal risk includes exposure to fines, penalti..."
2,What are the three operational risk measuremen...,9.1 Definition of Operational Risk \n 9.2 The...,The Supervisory Review,The three approaches are the Basic Indicator A...
3,What is the Basic Indicator Approach?,(c) Alpha = 15 per cent \n \n \n9.3.4 As a poi...,Banks using the Basic Indicator Approach are e...,The Basic Indicator Approach calculates operat...
4,What is the Standardised Approach?,risks are based on increasing risk sensitivity...,The Standardised Approach is one of the option...,The Standardised Approach calculates operation...
5,What is the Advanced Measurement Approach?,13.15 Quantitative and Qualitative Approa...,The Advanced Measurement Approach (AMA) is one...,The Advanced Measurement Approach allows banks...
6,How does HDFC define operational risk?,- 105 - \n \n9. Capital Charge for Operat...,Operational risk is defined as the risk of los...,HDFC defines operational risk as the risk aris...
7,What risks are excluded from operational risk?,- 105 - \n \n9. Capital Charge for Operat...,Operational risk is defined as the risk of los...,Strategic risk and reputational risk are exclu...
8,What is Basel III?,for the supervisory review process (Pillar 2) ...,Basel III is a set of international banking re...,Basel III is an international regulatory frame...
9,What is CET1 capital?,- 11 - \n4.2.2(viii). \n \n(ii) Common Equity ...,Common Equity Tier 1 (CET1) capital is a measu...,Common Equity Tier 1 (CET1) capital primarily ...


In [ ]:
df.to_csv("rag_results.csv", index=False)

print("Updated CSV saved successfully!")

Updated CSV saved successfully!


In [ ]:
from google.colab import files

files.download("rag_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>